In [15]:
import pandas as pd
import numpy as np

from sklearn.feature_extraction.text import CountVectorizer, TfidfVectorizer

from sklearn.naive_bayes import MultinomialNB
from sklearn.linear_model import LogisticRegression
from sklearn.svm import LinearSVC
from sklearn.ensemble import RandomForestClassifier

from sklearn.metrics import accuracy_score, classification_report, f1_score

In [16]:
train_df = pd.read_csv("../data/train.csv")
test_df = pd.read_csv("../data/test.csv")

# force clean
train_df = train_df.dropna(subset=['headline', 'category']).reset_index(drop=True)
test_df = test_df.dropna(subset=['headline', 'category']).reset_index(drop=True)

X_train = train_df['headline'].astype(str)
y_train = train_df['category']

X_test = test_df['headline'].astype(str)
y_test = test_df['category']

print(X_train.isna().sum())
print(X_test.isna().sum())

0
0


In [17]:
print(train_df['headline'].isnull().sum())
print(test_df['headline'].isnull().sum())

0
0


In [18]:
vectorizers = {
    "BoW" : CountVectorizer(max_features=5000),
    "TF-IDF": TfidfVectorizer(max_features=5000)
}

In [19]:
models = {
    "Naive Bayes": MultinomialNB(),
    "Logistic Regression": LogisticRegression(max_iter=1000),
    "Linear SVM": LinearSVC(),
    "Random Forest": RandomForestClassifier(random_state=42)
}

In [20]:
X_train.isnull().sum()

np.int64(0)

In [21]:
result = []

for vec_name, vectorizer in vectorizers.items():
    
    #transform text
    X_train_vec = vectorizer.fit_transform(X_train)
    X_test_vec = vectorizer.transform(X_test)

    for model_name, model in models.items():

        #train model
        model.fit(X_train_vec, y_train)

        #predict
        y_pred = model.predict(X_test_vec)

        #metrics
        acc = accuracy_score(y_test, y_pred)
        f1 = f1_score(y_test, y_pred, average='weighted')

        result.append({
            "vectorizers": vec_name,
            "Model": model_name,
            "Accuracy": round(acc, 4),
            "F1 Score": round(f1, 4)
        })

In [22]:
results_df = pd.DataFrame(result)

results_df = results_df.sort_values(by="Accuracy", ascending=False)

results_df

,vectorizers,Model,Accuracy,F1 Score
6,TF-IDF,Linear SVM,0.8239,0.8187
1,BoW,Logistic Regression,0.8183,0.8125
2,BoW,Linear SVM,0.8161,0.8124
5,TF-IDF,Logistic Regression,0.8157,0.8053
0,BoW,Naive Bayes,0.8138,0.8112
7,TF-IDF,Random Forest,0.7871,0.7758
4,TF-IDF,Naive Bayes,0.7799,0.7521
3,BoW,Random Forest,0.7782,0.7702


In [23]:
best_row = results_df.iloc[0]

print("Best Combination:")
print(best_row)

Best Combination:
vectorizers        TF-IDF
Model          Linear SVM
Accuracy           0.8239
F1 Score           0.8187
Name: 6, dtype: object


In [24]:
best_vectorizer_name = best_row['vectorizers']
best_model_name = best_row['Model']

print(best_vectorizer_name)
print(best_model_name)

TF-IDF
Linear SVM


In [25]:
if best_vectorizer_name == "BoW":
    best_vectorizer = CountVectorizer(max_features=5000)
else:
    best_vectorizer = TfidfVectorizer(max_features=5000)

In [26]:
if best_model_name == "Naive Bayes":
    best_model = MultinomialNB()

elif best_model_name == "Logistic Regression":
    best_model = LogisticRegression(max_iter=1000)

elif best_model_name == "Linear SVM":
    best_model = LinearSVC()

else:
    best_model = RandomForestClassifier(random_state=42)

In [27]:
X_train_vec = best_vectorizer.fit_transform(X_train)
X_test_vec = best_vectorizer.transform(X_test)

best_model.fit(X_train_vec, y_train)

final_pred = best_model.predict(X_test_vec)

print("FINAL MODEL ACCURACY:", accuracy_score(y_test, final_pred))
print(classification_report(y_test, final_pred))

FINAL MODEL ACCURACY: 0.8239112227805695
               precision    recall  f1-score   support

     Business       0.67      0.60      0.63      1198
Entertainment       0.85      0.86      0.86      3473
     Politics       0.85      0.92      0.89      7120
      Science       0.71      0.62      0.66       441
       Sports       0.82      0.71      0.76      1015
   Technology       0.69      0.47      0.56       421
        World       0.70      0.53      0.60       660

     accuracy                           0.82     14328
    macro avg       0.76      0.67      0.71     14328
 weighted avg       0.82      0.82      0.82     14328



In [28]:
import joblib
import os

os.makedirs("../models", exist_ok=True)

joblib.dump(best_model, "../models/best_model.pkl")
joblib.dump(best_vectorizer, "../models/best_vectorizer.pkl")

print("Model saved successfully!")

Model saved successfully!
